# Case Study: OEM1 Emissions Investigation

## Objective

The objective of this case study is to identify all OEM1 vehicles affected by the potentially defective T2 control units and determine the municipality in which these vehicles were registered.

According to the investigation information, affected T2 control units were produced by manufacturer 202 in plant 2022 between April 2009 and November 2014.

In addition, control units produced by manufacturer 201 in plant 2011 are affected when their production numbers range from 1250 to 19500.

The affected T2 control units are installed in OEM1 engines. These engines can be installed in OEM1 Type11 and Type12 vehicles.

The analysis therefore follows the supply chain:

**T2 control units → K1 engine components → OEM1 Type11/Type12 vehicles → vehicle registrations → municipality**

The available data are inspected and combined to create the final set of affected registered vehicles.

## 1. Data Selection

The available database contains information about individual parts, components, vehicles, registrations, geodata and logistics.

For this case study, the relevant datasets are:

- `Einzelteil_T02.txt` – production data for T2 control units
- `Bestandteile_Komponente_K1BE1.csv`
- `Bestandteile_Komponente_K1BE2.csv`
- `Bestandteile_Komponente_K1DI1.csv`
- `Bestandteile_Komponente_K1DI2.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ11.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ12.csv`
- `Zulassungen_alle_Fahrzeuge.csv`

The K1 component tables are required because they contain the relationship between T2 control units and the K1 engine components.

The OEM1 vehicle tables are required because they contain the relationship between the engine component and the vehicle.

Finally, the registration table is required to determine the municipality in which an affected vehicle was registered.

## 2. Import and Prepare the T2 Data

The original `Einzelteil_T02.txt` file contains records separated by tab characters.

The checkpoint analysis showed that the file contains two sets of T2 records. To make the file readable as a normal table, the tab characters are replaced by line breaks and the resulting file is read using whitespace separation.

The resulting dataset contains the T2 identifier, production date, manufacturer, production plant and defect information.

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("data/IDA SoSe26 - Data")

input_file = (data_path / "Einzelteil" / "Einzelteil_T02.txt")
fixed_file = (data_path / "Einzelteil" / "Einzelteil_T02_fixed.txt")
print(f"Input file: {input_file}")
print(f"Fixed file: {fixed_file}")

with open(input_file, "r", encoding="utf-8") as src, \
     open(fixed_file, "w", encoding="utf-8") as dst:

    while chunk := src.read(10_000_000):
        dst.write(chunk.replace("\t", "\n"))

df_t02 = pd.read_csv(
    fixed_file,
    sep=r"\s+",
    quotechar='"',
    na_values="NA"
)

print("Shape:", df_t02.shape)
print(df_t02.columns.tolist())
display(df_t02.head())

Input file: data/IDA SoSe26 - Data/Einzelteil/Einzelteil_T02.txt
Fixed file: data/IDA SoSe26 - Data/Einzelteil/Einzelteil_T02_fixed.txt


/tmp/ipykernel_3588/443950528.py:17: DtypeWarning: Columns (2,3,7,9,10,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_t02 = pd.read_csv(


Shape: (3204104, 15)
['X1', 'ID_T02.x', 'Produktionsdatum.x', 'Herstellernummer.x', 'Werksnummer.x', 'Fehlerhaft.x', 'Fehlerhaft_Datum.x', 'Fehlerhaft_Fahrleistung.x', 'ID_T02.y', 'Produktionsdatum.y', 'Herstellernummer.y', 'Werksnummer.y', 'Fehlerhaft.y', 'Fehlerhaft_Datum.y', 'Fehlerhaft_Fahrleistung.y']


,X1,ID_T02.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_T02.y,Produktionsdatum.y,Herstellernummer.y,Werksnummer.y,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y
1,4,2-201-2011-239,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,2-201-2011-304,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9,2-201-2011-125,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17,2-201-2011-55,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,19,2-201-2011-133,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Create a Unified T2 Dataset

The original T2 data contain two sets of columns (`.x` and `.y`).

The two sets are combined using `combine_first()` so that the final dataset contains one consistent set of T2 attributes.

This produces one row per T2 control unit with:

- T2 ID
- production date
- manufacturer
- production plant
- defect information
- defect date
- mileage at defect

In [2]:
t02 = pd.DataFrame({
    "ID_T02": df_t02["ID_T02.x"].combine_first(df_t02["ID_T02.y"]),
    "Produktionsdatum": df_t02["Produktionsdatum.x"].combine_first(
        df_t02["Produktionsdatum.y"]
    ),
    "Herstellernummer": df_t02["Herstellernummer.x"].combine_first(
        df_t02["Herstellernummer.y"]
    ),
    "Werksnummer": df_t02["Werksnummer.x"].combine_first(
        df_t02["Werksnummer.y"]
    ),
    "Fehlerhaft": df_t02["Fehlerhaft.x"].combine_first(
        df_t02["Fehlerhaft.y"]
    ),
    "Fehlerhaft_Datum": df_t02["Fehlerhaft_Datum.x"].combine_first(
        df_t02["Fehlerhaft_Datum.y"]
    ),
    "Fehlerhaft_Fahrleistung": df_t02[
        "Fehlerhaft_Fahrleistung.x"
    ].combine_first(
        df_t02["Fehlerhaft_Fahrleistung.y"]
    )
})

print(t02.shape)
display(t02.head())

(3204104, 7)


,ID_T02,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
1,2-201-2011-239,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904
2,2-201-2011-304,2008-11-07,201.0,2011.0,0.0,NaN,0.000000
3,2-201-2011-125,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904
4,2-201-2011-55,2008-11-07,201.0,2011.0,0.0,NaN,0.000000
5,2-201-2011-133,2008-11-07,201.0,2011.0,0.0,NaN,0.000000


## 4. Identify Affected T2 Control Units

Two groups of T2 control units are affected according to the case description.

### Group 1

Control units produced by:

- Manufacturer: **202**
- Plant: **2022**
- Production period: **April 2009 to November 2014**

### Group 2

Control units produced by:

- Manufacturer: **201**
- Plant: **2011**
- Production number: **1250 to 19500**

Both groups are identified separately and then combined into one affected T2 dataset.

In [3]:
t02_202 = t02[
    (t02["Herstellernummer"] == 202) &
    (t02["Werksnummer"] == 2022)
].copy()

t02_202["Produktionsdatum"] = pd.to_datetime(
    t02_202["Produktionsdatum"]
)

affected_202 = t02_202[
    (t02_202["Produktionsdatum"] >= "2009-04-01") &
    (t02_202["Produktionsdatum"] < "2014-12-01")
].copy()

print("Affected 202/2022:", affected_202.shape)

Affected 202/2022: (1589783, 7)


In [4]:
t02_201 = t02[
    (t02["Herstellernummer"] == 201) &
    (t02["Werksnummer"] == 2011)
].copy()

t02_201["Produktionsnummer"] = (
    t02_201["ID_T02"]
    .str.rsplit("-", n=1)
    .str[-1]
    .astype(int)
)

affected_201 = t02_201[
    t02_201["Produktionsnummer"].between(1250, 19500)
].copy()

print("Affected 201/2011:", affected_201.shape)

Affected 201/2011: (18251, 8)


In [5]:
affected_t2 = pd.concat(
    [affected_202, affected_201],
    ignore_index=True
)

print("Affected T2 units:", affected_t2.shape)
print("Unique T2 IDs:", affected_t2["ID_T02"].nunique())
print(
    "Duplicate T2 IDs:",
    affected_t2["ID_T02"].duplicated().sum()
)

Affected T2 units: (1608034, 8)
Unique T2 IDs: 1608034
Duplicate T2 IDs: 0


## 5. Trace Affected T2 Units to K1 Engine Components

The affected T2 units are installed in K1 engine components.

Per the general case-study brief, affected T2 units were installed in
Gasoline engines. We therefore restrict this analysis to the gasoline engine components:

- K1BE1 (Benzin / Gasoline)
- K1BE2 (Benzin / Gasoline)

(We discard following diesel engines components: K1DI1, K1DI2)

Each table contains an `ID_T2` column and the corresponding K1 component ID.

The affected T2 IDs are therefore used to filter each K1 dataset.

The resulting relationships are combined into one mapping table between affected T2 units and K1 engine components.

Restricting columns on read keeps memory usage down given these files are 100MB+.

In [6]:
K1BE1_file = (data_path / "Komponente" / "Bestandteile_Komponente_K1BE1.csv")
K1BE2_file = (data_path / "Komponente" / "Bestandteile_Komponente_K1BE2.csv")

affected_t2_ids = set(affected_t2["ID_T02"])

def load_k1_affected(path, id_col, k1_type):
    """Load a K1 component parts-list file, restricted to columns needed
    for the T2 > K1 join, and filter to rows containing an affected T2."""
    df = pd.read_csv(
        path,
        sep=";",
        quotechar='"',
        usecols=["ID_T2", id_col],
        dtype=str
    )
    df = df[df["ID_T2"].isin(affected_t2_ids)].copy()
    df = df.rename(columns={id_col: "ID_K1"})
    df["K1_Type"] = k1_type
    return df[["ID_T2", "ID_K1", "K1_Type"]]

k1be1_affected = load_k1_affected(
    K1BE1_file, "ID_K1BE1", "K1BE1"
)
k1be2_affected = load_k1_affected(
    K1BE2_file, "ID_K1BE2", "K1BE2"
)

affected_t2_k1 = pd.concat(
    [k1be1_affected, k1be2_affected],
    ignore_index=True
)

print("Affected T2 > K1 mappings:", affected_t2_k1.shape)
print(affected_t2_k1["K1_Type"].value_counts())
print("Unique T2 matched:", affected_t2_k1["ID_T2"].nunique())
print("Duplicate T2 matches:", affected_t2_k1["ID_T2"].duplicated().sum())
display(affected_t2_k1.head())

Affected T2 > K1 mappings: (803377, 3)
K1_Type
K1BE1    597655
K1BE2    205722
Name: count, dtype: int64
Unique T2 matched: 803377
Duplicate T2 matches: 0


,ID_T2,ID_K1,K1_Type
0,2-201-2011-1301,K1BE1-101-1011-151,K1BE1
1,2-201-2011-1271,K1BE1-101-1011-155,K1BE1
2,2-201-2011-1300,K1BE1-104-1041-374,K1BE1
3,2-201-2011-1252,K1BE1-101-1011-252,K1BE1
4,2-201-2011-1323,K1BE1-101-1011-256,K1BE1


## 6. Identify Affected OEM1 Vehicles

The OEM1 vehicle parts lists contain the engine component installed in each vehicle in the column `ID_Motor`.

Because the affected K1 components are the affected engine components, vehicles are identified by checking whether their `ID_Motor` occurs in the set of affected K1 IDs.

The analysis is performed separately for:

- OEM1 Type11
- OEM1 Type12

The two vehicle datasets are then combined.

In [7]:
type11_file = (data_path / "Fahrzeug" / "Bestandteile_Fahrzeuge_OEM1_Typ11.csv")
type12_file = (data_path / "Fahrzeug" / "Bestandteile_Fahrzeuge_OEM1_Typ12.csv")

vehicles_11 = pd.read_csv(
    type11_file,
    sep=";",
    quotechar='"',
    usecols=["ID_Motor", "ID_Fahrzeug"],
    dtype=str
)

vehicles_12 = pd.read_csv(
    type12_file,
    sep=";",
    quotechar='"',
    usecols=["ID_Motor", "ID_Fahrzeug"],
    dtype=str
)

affected_vehicles = pd.concat(
    [vehicles_11, vehicles_12],
    ignore_index=True
)

affected_k1_ids = set(affected_t2_k1["ID_K1"])

affected_vehicles_11 = vehicles_11[
    vehicles_11["ID_Motor"].isin(affected_k1_ids)
].copy()
affected_vehicles_11["Vehicle_Type"] = "Type11"

affected_vehicles_12 = vehicles_12[
    vehicles_12["ID_Motor"].isin(affected_k1_ids)
].copy()
affected_vehicles_12["Vehicle_Type"] = "Type12"

affected_vehicles = pd.concat(
    [affected_vehicles_11, affected_vehicles_12],
    ignore_index=True
)

print("Affected Type11 vehicles:", affected_vehicles_11.shape)
print("Affected Type12 vehicles:", affected_vehicles_12.shape)
print("Total affected vehicles:", affected_vehicles.shape)
print("Unique vehicle IDs:", affected_vehicles["ID_Fahrzeug"].nunique())
print("Duplicate vehicle IDs:", affected_vehicles["ID_Fahrzeug"].duplicated().sum())
display(affected_vehicles.head())

Affected Type11 vehicles: (495339, 3)
Affected Type12 vehicles: (102316, 3)
Total affected vehicles: (597655, 3)
Unique vehicle IDs: 597655
Duplicate vehicle IDs: 0


,ID_Motor,ID_Fahrzeug,Vehicle_Type
0,K1BE1-104-1041-536,11-1-11-273,Type11
1,K1BE1-104-1041-760,11-1-11-432,Type11
2,K1BE1-104-1041-515,11-1-11-520,Type11
3,K1BE1-104-1041-374,11-1-11-580,Type11
4,K1BE1-102-1021-97,11-1-11-615,Type11


## 7. Link Affected Vehicles to Registration Data

The registration dataset contains:

- `IDNummer` – vehicle identifier
- `Gemeinden` – municipality
- `Zulassung` – registration date

The vehicle identifier in the production data is `ID_Fahrzeug`.

Therefore, the registration data are linked using:

`ID_Fahrzeug = IDNummer`

The registration table contains unique vehicle identifiers, so a one-to-one merge is expected.

In [8]:
registrations_file = data_path / "Zulassungen" / "Zulassungen_alle_Fahrzeuge.csv"

registrations = pd.read_csv(
    registrations_file,
    sep=";",
    quotechar='"'
)

print("Registration rows:", len(registrations))
print("Unique IDNummer:", registrations["IDNummer"].nunique())
print("Duplicate IDNummer:",registrations["IDNummer"].duplicated().sum())
display(registrations.head())

Registration rows: 3204104
Unique IDNummer: 3204104
Duplicate IDNummer: 0


,Unnamed: 0,IDNummer,Gemeinden,Zulassung
0,408097,11-1-11-1,DRESDEN,2009-01-01
1,408098,11-1-11-2,DRESDEN,2009-01-01
2,1,12-1-12-1,LEIPZIG,2009-01-01
3,2,12-1-12-2,LEIPZIG,2009-01-01
4,3,12-1-12-3,DORTMUND,2009-01-01


In [9]:
registrations_small = registrations[
    ["IDNummer", "Gemeinden", "Zulassung"]
].copy()

try:
    affected_registered = affected_vehicles.merge(
        registrations_small,
        left_on="ID_Fahrzeug",
        right_on="IDNummer",
        how="left",
        validate="one_to_one"
    )
except Exception as e:
    print("Merge validation FAILED:", e)
    raise

print(affected_registered.shape)
display(affected_registered.head())

(597655, 6)


,ID_Motor,ID_Fahrzeug,Vehicle_Type,IDNummer,Gemeinden,Zulassung
0,K1BE1-104-1041-536,11-1-11-273,Type11,11-1-11-273,LUGAU/ERZGEB.,2009-01-02
1,K1BE1-104-1041-760,11-1-11-432,Type11,11-1-11-432,SUEDBROOKMERLAND,2009-01-02
2,K1BE1-104-1041-515,11-1-11-520,Type11,11-1-11-520,HEMMINGEN,2009-01-02
3,K1BE1-104-1041-374,11-1-11-580,Type11,11-1-11-580,HABICHTSWALD,2009-01-02
4,K1BE1-102-1021-97,11-1-11-615,Type11,11-1-11-615,BURGWALD,2009-01-02


In [10]:
print("affected_vehicles shape:", affected_vehicles.shape)
print(affected_vehicles["Vehicle_Type"].value_counts())
print()

# Break down by which K1 family the matched engine came from
k1_type_lookup = affected_t2_k1.set_index("ID_K1")["K1_Type"]
affected_vehicles["K1_Type"] = affected_vehicles["ID_Motor"].map(k1_type_lookup)
print(affected_vehicles["K1_Type"].value_counts(dropna=False))

affected_vehicles shape: (597655, 3)
Vehicle_Type
Type11    495339
Type12    102316
Name: count, dtype: int64

K1_Type
K1BE1    597655
Name: count, dtype: int64


**Note on K1BE2:** Although K1BE2 engine components were included in the T2 → K1 mapping (Step 5) for completeness, verification shows that OEM1 \
Type11 and Type12 vehicles exclusively use K1BE1 (gasoline) or K1DI1 (diesel) engines — K1BE2 and K1DI2 do not appear in either vehicle type's \
parts list. K1BE2 is therefore expected to contribute zero matches to the final affected-vehicle count, which is confirmed empirically below. 

In [11]:
# What do affected K1BE2 engine IDs actually look like?
k1be2_ids_affected = affected_t2_k1[affected_t2_k1["K1_Type"] == "K1BE2"]["ID_K1"]
print("Sample affected K1BE2 IDs:")
print(k1be2_ids_affected.head(10).tolist())
print()

# What ID_Motor values actually exist in the vehicle files, for comparison?
print("Sample ID_Motor values in Typ11:")
print(vehicles_11["ID_Motor"].head(10).tolist())
print()

# Direct check: does ANY K1BE2 engine (affected or not) ever appear as an ID_Motor
# in either vehicle file? This tests independent of the "affected" filter.
all_motor_ids_11 = set(vehicles_11["ID_Motor"])
all_motor_ids_12 = set(vehicles_12["ID_Motor"])
all_motor_ids = all_motor_ids_11 | all_motor_ids_12

k1be2_prefix_matches = [m for m in list(all_motor_ids)[:50000] if m.startswith("K1BE2")]
print("Sample of any K1BE2-prefixed engine IDs found in vehicle files:", k1be2_prefix_matches[:10])
print("Count found in this 50k sample:", len(k1be2_prefix_matches))

# More reliable: check via the K1BE2 component file itself
k1be2_component_file = data_path / "Komponente" / "Komponente_K1BE2.csv"
k1be2_all = pd.read_csv(k1be2_component_file, sep=";", quotechar='"', usecols=lambda c: True, nrows=5)
print()
print("Komponente_K1BE2 columns:", k1be2_all.columns.tolist())

Sample affected K1BE2 IDs:
['K1BE2-104-1041-210', 'K1BE2-104-1041-219', 'K1BE2-101-1011-73', 'K1BE2-101-1011-77', 'K1BE2-101-1011-78', 'K1BE2-104-1041-236', 'K1BE2-104-1041-252', 'K1BE2-104-1041-274', 'K1BE2-104-1041-310', 'K1BE2-104-1041-319']

Sample ID_Motor values in Typ11:
['K1BE1-101-1011-7', 'K1BE1-101-1011-12', 'K1BE1-101-1011-38', 'K1BE1-101-1011-97', 'K1BE1-101-1011-65', 'K1BE1-101-1011-53', 'K1BE1-104-1041-118', 'K1BE1-101-1011-100', 'K1BE1-101-1011-95', 'K1BE1-101-1011-50']

Sample of any K1BE2-prefixed engine IDs found in vehicle files: []
Count found in this 50k sample: 0

Komponente_K1BE2 columns: ['Unnamed: 0', 'X1', 'ID_Motor', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft', 'Fehlerhaft_Datum', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum_Origin_01011970', 'origin']


In [ ]:
# Extract the engine-family prefix from every ID_Motor value in each vehicle file
vehicles_11["engine_prefix"] = vehicles_11["ID_Motor"].str.split("-").str[0]
vehicles_12["engine_prefix"] = vehicles_12["ID_Motor"].str.split("-").str[0]

print("Engine family distribution in Typ11:")
print(vehicles_11["engine_prefix"].value_counts())
print()
print("Engine family distribution in Typ12:")
print(vehicles_12["engine_prefix"].value_counts())

# What engine family are the Type12 matches actually using?
matched_12 = affected_vehicles[affected_vehicles["Vehicle_Type"] == "Type12"]
matched_12_prefix = matched_12["ID_Motor"].str.split("-").str[0]
print("Engine family of matched Type12 vehicles:")
print(matched_12_prefix.value_counts())

## 8. Validate the Registration Merge

The registration merge is checked for vehicles for which no municipality was found.

A successful merge should result in no affected vehicles with a missing municipality.

In [ ]:
print(
    "Affected vehicles without registration:",
    affected_registered["Gemeinden"].isna().sum()
)

print(
    "Affected vehicles with registration:",
    affected_registered["Gemeinden"].notna().sum()
)

Affected vehicles without registration: 0
Affected vehicles with registration: 597655


## 9. Affected Vehicles by Municipality

The final registered vehicle dataset can now be aggregated by municipality.

This provides the number of affected OEM1 vehicles registered in each municipality.

In [ ]:
municipality_counts = (
    affected_registered["Gemeinden"]
    .value_counts()
    .rename_axis("Gemeinde")
    .reset_index(name="Affected_Vehicles")
)
print("Number of municipalities:", len(municipality_counts))
print(municipality_counts.head(20))

Number of municipalities: 5454
                Gemeinde  Affected_Vehicles
0                  KOELN              15232
1               DORTMUND               9571
2                LEIPZIG               7810
3                DRESDEN               7578
4                 BOCHUM               6672
5              BIELEFELD               5762
6                   BONN               5053
7              MUENSTER1               4799
8               MUENSTER               4731
9               AUGSBURG               4503
10         GELSENKIRCHEN               4441
11          BRAUNSCHWEIG               4310
12                AACHEN               3658
13                 MAINZ               3256
14            LEVERKUSEN               3029
15                 HERNE               2792
16  FREIBURG IM BREISGAU               2662
17            OSNABRUECK               2600
18               BOTTROP               2408
19            REGENSBURG               2392


In [ ]:
# Check for any municipality names with a trailing digit — possible
# artifact of a deduplication or export process upstream.
suspicious = registrations["Gemeinden"].str.contains(r"\d$", regex=True, na=False)
print("Municipalities with trailing digits:")
print(registrations.loc[suspicious, "Gemeinden"].value_counts())
print()

# Compare directly: how many total registrations for MUENSTER vs MUENSTER1?
print(registrations["Gemeinden"].value_counts().loc[["MUENSTER", "MUENSTER1"]])

Municipalities with trailing digits:
Gemeinden
MUENSTER1        25056
OBERHAUSEN2      13534
OBERHAUSEN1      13534
OBERHAUSEN3      13534
ALSDORF2          3640
                 ...  
LIND1                2
KARSTAEDT1           2
WERDER1              2
MARL1                2
GOENNERSDORF1        2
Name: count, Length: 343, dtype: int64

Gemeinden
MUENSTER     25056
MUENSTER1    25056
Name: count, dtype: int64


In [ ]:
geodata = pd.read_csv(
    data_path / "Geodaten" / "Geodaten_Gemeinden_v1.2_2017-08-22_TrR.csv",
    sep=";", quotechar='"'
)

print("Geodaten columns:", geodata.columns.tolist())
print()
print("Does 'MUENSTER1' exist in Geodaten?", "MUENSTER1" in geodata["Gemeinde"].values)
print("Does 'MUENSTER' exist in Geodaten?", "MUENSTER" in geodata["Gemeinde"].values)
print()
# Also check for ANY trailing-digit municipality names in geodata
suspicious_geo = geodata["Gemeinde"].str.contains(r"\d$", regex=True, na=False)
print("Trailing-digit municipalities in Geodaten:")
print(geodata.loc[suspicious_geo, "Gemeinde"].tolist())

Geodaten columns: ['Unnamed: 0', 'X', 'Postleitzahl', 'Gemeinde', 'Laengengrad', 'Breitengrad']

Does 'MUENSTER1' exist in Geodaten? True
Does 'MUENSTER' exist in Geodaten? True

Trailing-digit municipalities in Geodaten:
['HIRSCHFELD1', 'RUECKERSDORF1', 'SCHWARZBACH1', 'HARTMANNSDORF1', 'HEIDELAND1', 'PETERSBERG1', 'MERTENDORF1', 'HERMSDORF1', 'BUCHA1', 'LEHESTEN1', 'REINSDORF1', 'HIRSCHFELD2', 'HARTMANNSDORF2', 'BERNSDORF1', 'HAINICHEN1', 'GOLZOW1', 'FRIEDLAND1', 'SCHOENFELD1', 'MITTENWALDE1', 'SCHOENFELD2', 'GRUENOW1', 'BLANKENSEE1', 'ZIETHEN1', 'NEUENKIRCHEN1', 'NEUENKIRCHEN2', 'PAPENDORF1', 'DREETZ1', 'LOHMEN1', 'LOEBNITZ1', 'WUSTROW1', 'KUMMEROW1', 'LUESSOW1', 'STEINHAGEN1', 'WEITENHAGEN1', 'NEUENKIRCHEN3', 'GOEHREN1', 'PINNOW1', 'GRAMBOW1', 'KOENIGSFELD1', 'MOELLENBECK1', 'KARSTAEDT1', 'PASSOW1', 'WERDER1', 'BLANKENBERG1', 'HEIDENAU1', 'BASEDOW1', 'GUELZOW1', 'ELMENHORST1', 'WOLTERSDORF1', 'TRAMM1', 'NEUENKIRCHEN4', 'NEUENKIRCHEN5', 'SCHOENBERG1', 'HAMFELDE1', 'KOETHEL1', 'RAUSD

In [ ]:
print(geodata[geodata["Gemeinde"].isin(["MUENSTER", "MUENSTER1"])])

      Unnamed: 0     X  Postleitzahl   Gemeinde Laengengrad Breitengrad
2898        2899  2899         48143   MUENSTER    7,626804   51,961908
6363        6364  6364         86692  MUENSTER1   10,906217   48,623205


In [ ]:
geodata_small = geodata[["Gemeinde", "Postleitzahl", "Laengengrad", "Breitengrad"]].copy()

final_dataset = affected_registered.merge(
    geodata_small,
    left_on="Gemeinden",
    right_on="Gemeinde",
    how="left"
)

print(final_dataset.shape)
print("Rows without matched geodata:", final_dataset["Gemeinde"].isna().sum())
display(final_dataset.head())

(597655, 10)
Rows without matched geodata: 36


,ID_Motor,ID_Fahrzeug,Vehicle_Type,IDNummer,Gemeinden,Zulassung,Gemeinde,Postleitzahl,Laengengrad,Breitengrad
0,K1BE1-104-1041-536,11-1-11-273,Type11,11-1-11-273,LUGAU/ERZGEB.,2009-01-02,LUGAU/ERZGEB.,9385.0,"12,746491","50,739952"
1,K1BE1-104-1041-760,11-1-11-432,Type11,11-1-11-432,SUEDBROOKMERLAND,2009-01-02,SUEDBROOKMERLAND,26624.0,"7,357547","53,490698"
2,K1BE1-104-1041-515,11-1-11-520,Type11,11-1-11-520,HEMMINGEN,2009-01-02,HEMMINGEN,30966.0,"9,747743","52,32069"
3,K1BE1-104-1041-374,11-1-11-580,Type11,11-1-11-580,HABICHTSWALD,2009-01-02,HABICHTSWALD,34317.0,"9,310504","51,32613"
4,K1BE1-102-1021-97,11-1-11-615,Type11,11-1-11-615,BURGWALD,2009-01-02,BURGWALD,35099.0,"8,81134","51,027298"


In [ ]:
# For each unmatched name, check for near-matches in geodata
# (e.g., trailing whitespace, different suffix, alternate spelling)
unmatched = final_dataset[final_dataset["Gemeinde"].isna()]
print(unmatched["Gemeinden"].value_counts())
unmatched_names = unmatched["Gemeinden"].unique()
print("Unmatched municipality names:", unmatched_names)
print()

for name in unmatched_names:
    print(f"--- '{name}' ---")
    # exact match check (repr shows hidden whitespace)
    print("repr:", repr(name))
    # substring match in geodata, in case of partial name differences
    close = geodata[geodata["Gemeinde"].str.contains(name.strip(), case=False, na=False, regex=False)]
    print(close[["Gemeinde", "Postleitzahl"]].to_string())
    print()

Gemeinden
SEEG    36
Name: count, dtype: int64
Unmatched municipality names: ['SEEG']

--- 'SEEG' ---
repr: 'SEEG'
Empty DataFrame
Columns: [Gemeinde, Postleitzahl]
Index: []



**Note on unmatched geodata:** 36 of 597,655 affected, registered vehicles (0.006%) are registered in `SEEG`, a municipality not present in the \
provided geodata reference table (`Geodaten_Gemeinden_v1.2_2017-08-22`). These rows are retained in the final dataset with missing coordinate \
values, and are excluded only from the app's map visualization — they remain fully represented in the tabular data view and all non-geographic \
analysis.

In [ ]:
output_path = "data/SoSe26_Case_Study_finalData_Group_16.csv"

final_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", final_dataset.shape)
display(final_dataset.head())

NameError: name 'final_dataset' is not defined